# 06 - Visualization

Turns the baseline layer x region accuracy table into a heatmap, to
answer Q2: does AlexNet's layer hierarchy mirror the brain's visual
hierarchy?

In [ ]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results_df = pd.read_csv("results/tables/alexnet/baseline_accuracy.csv")
results_df.head()

## Reshape into a layer x region matrix

Layers are ordered early to late (conv1 -> fc8) and regions are ordered
roughly early visual cortex to higher-level cortex, so any hierarchy
pattern is easy to see visually.

In [ ]:
layer_order = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7", "fc8"]
region_order = ["V1", "V2", "V3", "V4", "LOC", "EBA", "FFA", "STS", "PPA"]

pivot = results_df.pivot(index="layer", columns="region", values="accuracy")
pivot = pivot.reindex(index=layer_order, columns=region_order)
pivot

## Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")

ax.set_xticks(range(len(region_order)))
ax.set_xticklabels(region_order, rotation=45, ha="right")
ax.set_yticks(range(len(layer_order)))
ax.set_yticklabels(layer_order)

ax.set_xlabel("Brain region")
ax.set_ylabel("AlexNet layer")
ax.set_title("Encoding accuracy: AlexNet layer vs. brain region")

for i in range(len(layer_order)):
    for j in range(len(region_order)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                color="white" if val < pivot.values.max() * 0.6 else "black",
                fontsize=8)

fig.colorbar(im, ax=ax, label="Mean correlation (accuracy)")
fig.tight_layout()

import os
os.makedirs("results/figures/alexnet", exist_ok=True)
fig.savefig("results/figures/alexnet/layer_region_heatmap.png", dpi=150)

plt.show()

## Best layer per region

For each region, which layer predicts it best? This is the clearest
single summary of whether the hierarchy pattern holds.

In [ ]:
best_layer_per_region = pivot.idxmax(axis=0)
best_accuracy_per_region = pivot.max(axis=0)

summary = pd.DataFrame({
    "best_layer": best_layer_per_region,
    "best_accuracy": best_accuracy_per_region,
})
summary = summary.reindex(region_order)
print(summary)

summary.to_csv("results/tables/alexnet/best_layer_per_region.csv")

## Save and commit

Run this from within Colab to push the figure and summary table back to
the repo.

In [ ]:
!git config --global user.email "your-email@example.com"
!git config --global user.name "sossyh"
!git add results/figures/alexnet/layer_region_heatmap.png results/tables/alexnet/best_layer_per_region.csv
!git commit -m "Add layer x region heatmap and best-layer summary"
!git push origin main